# Recovery modeling and tail risk

**Start here:** This deep dive expands on `07_advanced_quant/correlation_and_credit_models.ipynb`; read the overview first for the common copula, latent-factor, and recovery vocabulary.

Compare **constant**, **market-correlated**, and **market-standard stochastic** recovery via `RecoverySpec` / `RecoveryModel`, then stress **portfolio losses** when **LGD co-moves** with the same systematic factor as **defaults**.


## Concept

**Recovery** (or **LGD = 1 − R**) need not be fixed. **Market-correlated** recovery makes recovery depend on the **systematic** state: in bad markets, recoveries often **fall**, so **defaults** and **LGD** can **worsen together**.

`RecoveryModel.conditional_recovery(Z)` and **`conditional_lgd(Z)`** describe that slice. The canonical convention is **low factor = stress**; positive recovery-factor sensitivity therefore lowers recovery when $Z<0$. The market-standard preset uses this positive sensitivity.

We first tabulate conditional recovery/LGD across $Z \in [-3, 3]$, then pass the same `RecoverySpec` to the Rust portfolio-loss engine so defaults and factor-dependent LGD share each path's systematic factor.

## API walkthrough

- **`RecoverySpec.constant(rate)`** → **`.build()`** — fixed recovery; LGD is constant in $Z$.
- **`RecoverySpec.market_correlated(mean, vol, correlation)`** — Andersen–Sidenius-style systematic recovery.
- **`RecoverySpec.market_standard_stochastic()`** — preset mean/vol/correlation calibration.
- **`simulate_portfolio_loss(exposures, config, recovery_spec)`** — uses `conditional_lgd(Z)` on each default inside the Rust path simulation.

Built models expose **`expected_recovery`**, **`recovery_volatility`**, and **`is_stochastic`**.

In [1]:
from finstack_quant.valuations.correlation import (
    CopulaSpec,
    CreditExposure,
    PortfolioLossConfig,
    RecoverySpec,
    simulate_portfolio_loss,
)

spec_const = RecoverySpec.constant(0.40)
spec_corr = RecoverySpec.market_correlated(0.40, 0.10, 0.50)
spec_stoch = RecoverySpec.market_standard_stochastic()
rec_const = spec_const.build()
rec_corr = spec_corr.build()
rec_stoch = spec_stoch.build()

print(f"Constant: {rec_const}")
print(f"Market-correlated: {rec_corr}, vol={rec_corr.recovery_volatility:.4f}")
print(f"Market standard stochastic: {rec_stoch}, vol={rec_stoch.recovery_volatility:.4f}")
print(
    f"Unconditional expected recovery: const={rec_const.expected_recovery:.4f}, "
    f"corr={rec_corr.expected_recovery:.4f}, mkt_std={rec_stoch.expected_recovery:.4f}"
)

Constant: RecoveryModel('Constant Recovery', expected=0.4000)
Market-correlated: RecoveryModel('Market-Correlated Stochastic Recovery', expected=0.4010), vol=0.1000
Market standard stochastic: RecoveryModel('Market-Correlated Stochastic Recovery', expected=0.4039), vol=0.2500
Unconditional expected recovery: const=0.4000, corr=0.4010, mkt_std=0.4039


**Conditional** **recovery** and **LGD** across **market** $Z \in \{-3, -2, \ldots, 3\}$.

In [2]:
print("Z    | R_const | LGD_c | R_corr | LGD_corr | R_mktstd | LGD_mktstd")
for z in range(-3, 4):
    zf = float(z)
    print(
        f"{zf:4.1f} | {rec_const.conditional_recovery(zf):7.4f} | "
        f"{rec_const.conditional_lgd(zf):5.4f} | {rec_corr.conditional_recovery(zf):6.4f} | "
        f"{rec_corr.conditional_lgd(zf):8.4f} | {rec_stoch.conditional_recovery(zf):8.4f} | "
        f"{rec_stoch.conditional_lgd(zf):10.4f}"
    )
print("Bad states (negative Z in many credit conventions) often lift LGD when recovery is market-linked.")

Z    | R_const | LGD_c | R_corr | LGD_corr | R_mktstd | LGD_mktstd
-3.0 |  0.4000 | 0.6000 | 0.2630 |   0.7370 |   0.1604 |     0.8396
-2.0 |  0.4000 | 0.6000 | 0.3053 |   0.6947 |   0.2246 |     0.7754
-1.0 |  0.4000 | 0.6000 | 0.3512 |   0.6488 |   0.3053 |     0.6947
 0.0 |  0.4000 | 0.6000 | 0.4000 |   0.6000 |   0.4000 |     0.6000
 1.0 |  0.4000 | 0.6000 | 0.4509 |   0.5491 |   0.5028 |     0.4972
 2.0 |  0.4000 | 0.6000 | 0.5028 |   0.4972 |   0.6054 |     0.3946
 3.0 |  0.4000 | 0.6000 | 0.5547 |   0.4453 |   0.6994 |     0.3006
Bad states (negative Z in many credit conventions) often lift LGD when recovery is market-linked.


## Examples

Reuse the **10-name** portfolio from the default simulation notebook: same PDs, notionals, Gaussian copula, asset correlation **0.20**, and **10,000** path-indexed Philox streams with seed **42**. On each default, the Rust engine applies `conditional_lgd(Z)` from the active recovery specification; constant 40% recovery exactly matches 60% exposure LGD.

Print Rust-owned expected loss, nearest-rank VaR(99%), and ES(99%) for each recovery assumption.

In [3]:
pds = [0.02, 0.03, 0.05, 0.02, 0.04, 0.03, 0.06, 0.01, 0.03, 0.05]
rho = 0.20
n_sims = 10_000
exposures = [
    CreditExposure(
        id=f"name_{index + 1}",
        notional=10_000_000.0,
        default_probability=pd,
        lgd=0.60,
        factor_loadings=[rho**0.5],
    )
    for index, pd in enumerate(pds)
]
config = PortfolioLossConfig(n_sims, 42, 0.99, CopulaSpec.gaussian())
specs = [
    ("constant_40pct_R", spec_const),
    ("market_correlated", spec_corr),
    ("market_standard_stoch", spec_stoch),
]

results = {}
print(f"Portfolio sim: {n_sims} paths, seed=42, rho={rho}")
for label, spec in specs:
    result = simulate_portfolio_loss(exposures, config, spec)
    results[label] = result
    print(
        f"{label:22} mean={result.expected_loss:12,.0f}  "
        f"VaR99={result.var:12,.0f}  ES99={result.expected_shortfall:12,.0f}"
    )

base = results["constant_40pct_R"]
market = results["market_correlated"]
print(
    f"Tail amplification (market-corr vs constant): "
    f"VaR99 {((market.var / base.var) - 1) * 100:+.2f}%, "
    f"ES99 {((market.expected_shortfall / base.expected_shortfall) - 1) * 100:+.2f}%"
)

Portfolio sim: 10000 paths, seed=42, rho=0.2
constant_40pct_R       mean=   2,048,400  VaR99=  18,000,000  ES99=  21,623,762
market_correlated      mean=   2,212,997  VaR99=  20,833,147  ES99=  25,712,257
market_standard_stoch  mean=   2,355,602  VaR99=  23,248,299  ES99=  28,954,866
Tail amplification (market-corr vs constant): VaR99 +15.74%, ES99 +18.91%


## Takeaways

- **Three** recovery styles — constant, market-correlated, and market-standard stochastic — share the same `RecoveryModel` surface.
- `simulate_portfolio_loss(..., recovery_spec)` applies factor-dependent LGD inside the canonical Rust path, preserving the same systematic state used for defaults.
- When LGD rises in bad systematic states alongside higher conditional PDs, loss-positive VaR and ES can exceed the constant-LGD case.
- Always align the low-factor-stress sign convention across copula and recovery inputs.
- For full structured-credit workflows, combine with `clo_tranche_modeling.ipynb` and `portfolio_default_simulation.ipynb`.

### Named recovery model

`RecoverySpec.build()` returns a concrete `RecoveryModel` used by downstream portfolio loss simulations.

In [4]:
from finstack_quant.valuations.correlation import RecoveryModel

print("RecoveryModel type:", RecoveryModel.__name__)
print("constant recovery typed:", isinstance(rec_const, RecoveryModel))
print("market-correlated recovery typed:", isinstance(rec_corr, RecoveryModel))

RecoveryModel type: RecoveryModel
constant recovery typed: True
market-correlated recovery typed: True
